# 08c · Topology sweeps for the existing LOIO models (inference only, any GPU, ≈ 1–2 h)
**Block B**: writes `topo_test.csv` into each seed-0 LOIO run folder of the baseline, clDice and Skeleton Recall (60 models). The Tversky models get theirs in 08a/08b. Can run in parallel with 08a/08b. Resumable.
Specification: `specifications/03_spec_recall_control_topology.md` (written before these runs; SHA-256 in `specifications/README.md`).

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
from gwfss_stem import plan, engine
dirs = [os.path.join(RESULTS, engine.run_name({**engine.DEFAULTS, **{k: v for k, v in r.items() if k != 'sq'}})) for r in plan.loio_plan()]
print(sum(os.path.exists(f'{d}/final.pt') for d in dirs), 'of', len(dirs), 'models found')
for d in dirs:
    engine.topology_sweep(d, DATA_ROOT, SPLIT_FILE, subset='test')
print('done')